In [ ]:
import numpy as np
import pandas as pd
import joblib

INPUT_CSV = "FinalTestDataset2025.xls"        
OUTPUT_CSV = "PCRPredictions.csv"
MODEL_PKL = "catboost_fs_final.pkl"


def main():
    bundle = joblib.load(MODEL_PKL)

    model = bundle["model"]           
    cols_with_999 = bundle["cols_with_999"]
    id_col = bundle["id_col"]
    target_col = bundle["target_col"]
    threshold = bundle["threshold"]

    print(f"Loaded model bundle from: {MODEL_PKL}")
    print(f"Using threshold: {threshold}")

    df = pd.read_excel(INPUT_CSV)
    print(f"Loaded test data from: {INPUT_CSV}")
    print("Shape:", df.shape)

    cols999_present = [c for c in cols_with_999 if c in df.columns]
    if cols999_present:
        df[cols999_present] = df[cols999_present].replace(999, np.nan)
        print(f"Replaced 999 with NaN in columns: {cols999_present}")
    else:
        print("No 999-coded columns present in this dataset.")

    for col in cols999_present:
        flag_col = col + "_missing"
        if flag_col not in df.columns:
            df[flag_col] = df[col].isna().astype(int)

    drop_cols = [id_col]
    if target_col in df.columns:
        drop_cols.append(target_col)

    X_new = df.drop(columns=drop_cols, errors="ignore")
    print("Feature matrix shape:", X_new.shape)

    y_proba = model.predict_proba(X_new)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    if id_col not in df.columns:
        raise KeyError(f"ID column '{id_col}' not found in test data!")

    results = pd.DataFrame({
        id_col: df[id_col],
        "pCR_pred": y_pred,
    })

    print("Preview of predictions:")
    print(results.head())

    results.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved predictions to: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()


Loaded model bundle from: catboost_fs_final.pkl
Using threshold: 0.5
Loaded test data from: FinalTestDataset2025.xls
Shape: (133, 119)
Replaced 999 with NaN in columns: ['PgR', 'HER2', 'TrippleNegative', 'ChemoGrade', 'Proliferation', 'HistologyType', 'LNStatus', 'Gene']
Feature matrix shape: (133, 126)
Preview of predictions:
          ID  pCR_pred
0  TRG002219         0
1  TRG002222         0
2  TRG002223         0
3  TRG002235         0
4  TRG002240         0
Saved predictions to: PCRPredictions.csv
